# StoryWeaver — Qwen2.5-1.5B LoRA 推理（Colab T4）

**使用步骤：**
1. 顶部菜单 → **修改运行时类型** → 选择 **T4 GPU**
2. 将 `qwen2_5_1_5b_lora.zip` 上传到你的 Google Drive 根目录（或修改下方路径）
3. 按顺序运行所有单元格

In [ ]:
# ✅ 第一步：确认 GPU 可用
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() if result.returncode == 0 else '❌ 未检测到GPU，请检查运行时类型')

In [ ]:
# ✅ 第二步：安装依赖（兼容 Colab Python 3.12）
!pip install -q --upgrade "pip<26"
!pip install -q "transformers>=4.45.0" "accelerate>=0.26.0" "peft==0.14.0" "bitsandbytes>=0.45.2"
print('依赖安装完成（含 bitsandbytes 兼容版本）')
print('⚠️ 如果这是首次安装/升级依赖，建议 Runtime -> Restart runtime 后再继续')

In [ ]:
# ✅ 第三步：挂载 Google Drive 并解压模型（自动查找zip）
from google.colab import drive
import os, zipfile, glob

drive.mount('/content/drive')

EXTRACT_DIR = '/content/models'
os.makedirs(EXTRACT_DIR, exist_ok=True)

# 会按顺序尝试这些路径
candidate_paths = [
    '/content/drive/MyDrive/qwen2_5_1_5b_lora.zip',
    '/content/qwen2_5_1_5b_lora.zip',
    '/content/drive/MyDrive/nlp/qwen2_5_1_5b_lora.zip',
]

# 再自动扫描 MyDrive 根目录和一级子目录
candidate_paths.extend(glob.glob('/content/drive/MyDrive/*.zip'))
candidate_paths.extend(glob.glob('/content/drive/MyDrive/*/*.zip'))

# 仅保留 zip 且优先包含 qwen2_5_1_5b_lora 的文件
candidate_paths = [p for p in candidate_paths if p.endswith('.zip')]
preferred = [p for p in candidate_paths if 'qwen2_5_1_5b_lora' in os.path.basename(p)]
zip_path = preferred[0] if preferred else (candidate_paths[0] if candidate_paths else None)

if not zip_path or not os.path.exists(zip_path):
    raise FileNotFoundError(
        '未找到模型zip。请把 qwen2_5_1_5b_lora.zip 上传到 Drive 根目录，'
        '或放到 /content 后重试。'
    )

print('使用zip:', zip_path)
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(EXTRACT_DIR)

LORA_PATH = os.path.join(EXTRACT_DIR, 'qwen2_5_1_5b_lora')
if not os.path.isdir(LORA_PATH):
    # 兼容 zip 内部目录名不一致，自动找包含 adapter_model.safetensors 的目录
    found = None
    for root, dirs, files in os.walk(EXTRACT_DIR):
        if 'adapter_model.safetensors' in files and 'adapter_config.json' in files:
            found = root
            break
    if not found:
        raise FileNotFoundError('已解压，但未找到 LoRA 目录（缺少 adapter_model.safetensors）')
    LORA_PATH = found

print('模型路径:', LORA_PATH)
print('目录文件:', os.listdir(LORA_PATH))

In [ ]:
# ✅ 第四步：加载基础模型 + LoRA 适配器（含预热）
# 策略：float16 直接加载，T4(16GB) 完全够用

import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 先检查 bitsandbytes 是否可用（某些 Colab 环境会版本漂移）
try:
    import bitsandbytes  # noqa: F401
    print('bitsandbytes: OK')
except Exception:
    print('bitsandbytes 不可用，自动修复中...')
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'bitsandbytes>=0.45.2'])
    import bitsandbytes  # noqa: F401
    print('bitsandbytes 已修复；如仍异常请重启运行时再执行本格')

from peft import PeftModel

BASE_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('正在加载分词器...')
tokenizer = AutoTokenizer.from_pretrained(LORA_PATH, trust_remote_code=True)

print(f'正在加载基础模型 {BASE_MODEL} (float16)...')
t0 = time.time()
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
base.to(DEVICE)
print(f'基础模型加载完成，耗时 {time.time()-t0:.1f}s')

print('正在加载LoRA适配器...')
model = PeftModel.from_pretrained(base, LORA_PATH)
model.eval()
print('✅ 模型加载完成')
if DEVICE == 'cuda':
    print(f'✅ 当前显存占用: {torch.cuda.memory_allocated()/1e9:.2f} GB')
else:
    print('⚠️ 当前未使用GPU，请检查 Colab 运行时是否为 T4')

# 预热：避免首轮真实请求触发编译/缓存导致慢5秒+
print('开始模型预热（一次性）...')
warm_prompt = '请用一句话描述霍格沃茨的夜晚。'
warm_messages = [{'role': 'user', 'content': warm_prompt}]
warm_text = tokenizer.apply_chat_template(
    warm_messages, tokenize=False, add_generation_prompt=True
)
warm_inputs = tokenizer(warm_text, return_tensors='pt').to(DEVICE)
with torch.no_grad():
    _ = model.generate(
        **warm_inputs,
        max_new_tokens=12,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
print('✅ 预热完成，后续首答会明显更快')

In [ ]:
# ✅ 第五步：推理函数（2秒响应控制）
# T4 float16 实测约 40-50 tok/s
# FAST_MAX_TOKENS=60 → 约 1.5s | =80 → 约 2s

FAST_MAX_TOKENS = 60   # ⚙️ 调整此值控制输出长度/速度

def generate(prompt: str, max_new_tokens: int = FAST_MAX_TOKENS) -> str:
    messages = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors='pt').to(DEVICE)

    t_start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.8,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.time() - t_start

    input_len = inputs['input_ids'].shape[-1]
    new_tokens = outputs[0][input_len:]
    result = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    print(f'[生成完成] {len(new_tokens)} tokens | 耗时 {elapsed:.2f}s | 速度 {len(new_tokens)/elapsed:.1f} tok/s')
    return result

print('推理函数已定义，FAST_MAX_TOKENS =', FAST_MAX_TOKENS)

In [ ]:
# ✅ 第六步：高质量双阶段生成（2秒钩子 + 高质量重写）
from transformers import TextIteratorStreamer
from threading import Thread
import time

def _build_hook_prompt(user_prompt: str) -> str:
    return (
        '你是中文小说作家。请只写1句高质量悬念句，要求：\n'
        '1) 保持哈利波特奇幻氛围\n'
        '2) 不要总结，不要解释\n'
        '3) 只输出一句，不超过35个汉字\n\n'
        f'用户情节：{user_prompt}\n\n'
        '悬念句：'
    )

def _build_full_prompt(user_prompt: str, hook: str) -> str:
    return (
        '你是中文长篇奇幻小说作者。请基于以下情节续写，输出1-2段、约180-260字，要求：\n'
        '1) 文风自然、有画面感\n'
        '2) 避免套路台词和空泛说教\n'
        '3) 不要重复前句，不要自我解释\n'
        '4) 保持人物行为动机合理\n\n'
        f'情节：{user_prompt}\n'
        f'开场钩子：{hook}\n\n'
        '续写：'
    )

def _generate_text(prompt: str, max_new_tokens: int, temperature: float, top_p: float,
                   do_sample: bool = True, max_time: float = None, stream: bool = False) -> str:
    messages = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(DEVICE)

    gen_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=do_sample,
        repetition_penalty=1.08,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    if max_time is not None:
        gen_kwargs['max_time'] = max_time

    if stream:
        streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
        gen_kwargs['streamer'] = streamer
        t = Thread(target=model.generate, kwargs=gen_kwargs)
        t.start()
        chunks = []
        for piece in streamer:
            print(piece, end='', flush=True)
            chunks.append(piece)
        print()
        return ''.join(chunks).strip()

    with torch.no_grad():
        outputs = model.generate(**gen_kwargs)
    in_len = inputs['input_ids'].shape[-1]
    out_ids = outputs[0][in_len:]
    return tokenizer.decode(out_ids, skip_special_tokens=True).strip()

def quality_two_stage_generate(
    prompt: str,
    hook_time_budget: float = 2.0,
    hook_max_tokens: int = 28,
    full_max_tokens: int = 220,
    stream_full: bool = True,
    fallback_hook: str = '禁林深处，一道不属于人类的呼吸声贴近了他。',
) -> dict:
    # 阶段1：高质量短钩子（严格时间预算）
    print('【阶段1：2秒内返回高质量钩子】')
    hook_prompt = _build_hook_prompt(prompt)
    t0 = time.time()
    hook = _generate_text(
        prompt=hook_prompt,
        max_new_tokens=hook_max_tokens,
        temperature=0.55,
        top_p=0.9,
        do_sample=True,
        max_time=hook_time_budget,
        stream=False,
    ).strip()
    dt = time.time() - t0

    # 若模型在预算内输出为空，给业务兜底句，保证体验
    if not hook:
        hook = fallback_hook
    print(f'[阶段1完成] {dt:.2f}s')
    print('>>', hook)

    # 阶段2：不是“接着拼”，而是“基于钩子重写高质量正文”
    print('\n【阶段2：高质量续写（重写模式）】')
    full_prompt = _build_full_prompt(prompt, hook)
    if stream_full:
        full_text = _generate_text(
            prompt=full_prompt,
            max_new_tokens=full_max_tokens,
            temperature=0.82,
            top_p=0.92,
            do_sample=True,
            stream=True,
        )
    else:
        full_text = _generate_text(
            prompt=full_prompt,
            max_new_tokens=full_max_tokens,
            temperature=0.82,
            top_p=0.92,
            do_sample=True,
            stream=False,
        )

    return {
        'hook': hook,
        'full_text': full_text,
        'hook_latency_sec': dt,
    }

# 示例
prompt = '哈利波特走进禁林，感受到一股神秘的魔法气息。他小心翼翼地向前走去，突然——'
result = quality_two_stage_generate(
    prompt,
    hook_time_budget=2.0,
    hook_max_tokens=24,
    full_max_tokens=220,
    stream_full=True,
)

print('\n===== 结果汇总 =====')
print('首答时延:', f"{result['hook_latency_sec']:.2f}s")
print('首答钩子:', result['hook'])
print('\n最终正文（前1000字）：')
print(result['full_text'][:1000])

In [ ]:
# ✅ 第七步：启动 FastAPI 服务器（让后端通过 HTTP 调用模型）
# 一共两个接口：
#   POST /generate         —— 2秒限制短答
#   POST /two_stage        —— 高质量双阶段（钉子+续写）

import threading, uvicorn
from fastapi import FastAPI
from fastapi.responses import JSONResponse
from pydantic import BaseModel

# 安装服务端依赖
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'fastapi', 'uvicorn[standard]', 'pydantic'], check=True)

# 安装 cloudflared（无需登录，比 ngrok 更稳定）
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

app = FastAPI(title='StoryWeaver Qwen2.5-LoRA API')

class GenerateRequest(BaseModel):
    prompt: str
    max_new_tokens: int = 60

class TwoStageRequest(BaseModel):
    prompt: str
    hook_time_budget: float = 2.0
    hook_max_tokens: int = 24
    full_max_tokens: int = 220
    fallback_hook: str = '\u7981\u6797\u6df1\u5904\uff0c\u4e00\u9053\u4e0d\u5c5e\u4e8e\u4eba\u7c7b\u7684\u547c\u5438\u58f0\u8d34\u8fd1\u4e86\u4ed6\u3002'

@app.get('/health')
def health():
    return {'status': 'ok', 'model': 'Qwen2.5-1.5B-LoRA'}

@app.post('/generate')
def api_generate(req: GenerateRequest):
    """2秒内直接返回，适合后端快速调用"""
    try:
        text = generate(req.prompt, max_new_tokens=req.max_new_tokens)
        return {'text': text, 'prompt': req.prompt}
    except Exception as e:
        return JSONResponse(status_code=500, content={'error': str(e)})

@app.post('/two_stage')
def api_two_stage(req: TwoStageRequest):
    """\u9ad8\u8d28\u91cf\u53cc\u9636\u6bb5：\u8fd4\u56de hook + full_text"""
    try:
        result = quality_two_stage_generate(
            prompt=req.prompt,
            hook_time_budget=req.hook_time_budget,
            hook_max_tokens=req.hook_max_tokens,
            full_max_tokens=req.full_max_tokens,
            stream_full=False,   # API 模式不用流式
            fallback_hook=req.fallback_hook,
        )
        return result
    except Exception as e:
        return JSONResponse(status_code=500, content={'error': str(e)})

# 在后台线程里启动 uvicorn
def _run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='error')

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()
print('\u670d\u52a1\u5668\u5df2\u542f\u52a8 -> http://localhost:8000')
print('\u5f00\u59cb cloudflared \u5185\u7f51\u7a7f\u900f...')

# 启动 cloudflared 实时隔道，自动打印公网地址
import subprocess, re, time as _time
cf_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
print('\u7b49待分配公网地址...')
for line in cf_proc.stdout:
    m = re.search(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', line)
    if m:
        PUBLIC_URL = m.group(0)
        print('=' * 55)
        print(f'\u2705 \u516c\u7f51\u5730\u5740: {PUBLIC_URL}')
        print(f'   /generate  POST {PUBLIC_URL}/generate')
        print(f'   /two_stage POST {PUBLIC_URL}/two_stage')
        print(f'\u5c06\u6b64\u5730\u5740\u586b\u5165 .env \u7684 COLAB_MODEL_URL')
        print('=' * 55)
        break

## 双阶段输出：2秒内先给结果，再补充长续写

这个方案把生成拆成两段：
1. 第一段 `max_new_tokens=40~60`，保证约 1-2 秒内有可用内容
2. 第二段在后台继续生成更长内容，再拼接输出

你可以直接运行下一个代码单元格。

In [ ]:
# ✅ 第七步：测试生成（哈利波特主题）
prompt = '哈利波特走进禁林，感受到一股神秘的魔法气息。他小心翼翼地向前走去，突然——'
response = generate(prompt)
print('\n=== 模型输出 ===')
print(response)

In [ ]:
# ✅ 第八步：基准测试（测量真实速度）
import statistics

test_prompts = [
    '霍格沃茨城堡的大厅里，邓布利多教授说：',
    '赫敏从图书馆借来了一本古老的魔法书，她翻开第一页，',
    '罗恩骑上扫帚，飞越魁地奇球场，眼前出现了',
]

times = []
for i, p in enumerate(test_prompts):
    t = time.time()
    _ = generate(p)
    times.append(time.time() - t)

print(f'\n=== 速度汇总 ===')
print(f'平均耗时: {statistics.mean(times):.2f}s')
print(f'最大耗时: {max(times):.2f}s')
print(f'目标 2s 以内: {"✅ 达标" if statistics.mean(times) <= 2.0 else "⚠️ 未达标，可减小 FAST_MAX_TOKENS"}')

## 速度调优参考

| `FAST_MAX_TOKENS` | T4 float16 预估时间 | 适用场景 |
|:-:|:-:|:-:|
| 40 | ~1.0s | 极速单句 |
| **60** | **~1.5s** | **推荐（默认）** |
| 80 | ~2.0s | 较长段落 |
| 120 | ~3.0s | 完整段落 |

如果超时，在第五步减小 `FAST_MAX_TOKENS` 即可。